# T4：音訊較短造成尾幀截斷的回歸測試

只測試 **音訊尾端補靜音、完整保留影片幀數**。保留現有主角、追蹤 ID、Pose、DeepGaze 與構圖策略。
短暫變暗、人物交錯及 3D 慢動作運鏡可能影響追蹤 ID；最後一格列出分段診斷供對照，**不把 ID 缺失等同視覺焦點錯誤**。

請使用 **T4、Python 3.13+、已安裝本專案依賴**的 Colab，先修改第一格的三個路徑，再依序執行。
本 notebook 內附 runner 及最小修補，支援原基準版與已修正版 checkout，不重新安裝 PyTorch／CUDA。
修補只套用到已知版本的 `reframe/video_io.py`，遇到未知修改會停止。

先做兩秒 mux 短測，再跑 **一次完整 gap1**，沒有四組比较或重複暖機。
依前次 170 秒影片結果，完整推論約 **33 分鐘**，另計模型初始化／下載、檢查與 Drive 備份。


In [ ]:
from datetime import datetime, timezone
from pathlib import Path
import hashlib, json, shutil, subprocess, sys, zipfile
from google.colab import drive

assert sys.version_info >= (3, 13), "需要 Python 3.13+"
drive.mount("/content/drive")
repo = Path("/content/auto-vertical-reframe")
VIDEO_SOURCE = Path("/content/drive/MyDrive/crop/vu110.mp4")  # 修改這裡
SEG_MODEL = Path("/content/drive/MyDrive/crop/yolo26n-seg.pt")
POSE_MODEL = Path("/content/drive/MyDrive/crop/yolo26n-pose.pt")
assert (repo / "scripts/benchmark_colab.py").is_file(), "請先 clone 並依 README 安裝本專案"
for path in (VIDEO_SOURCE, SEG_MODEL, POSE_MODEL):
    assert path.is_file(), str(path)
subprocess.run(["nvidia-smi"], check=True)
subprocess.run([sys.executable, "-c",
    "import cv2, numpy, torch, torchvision, ultralytics, scenedetect, lap; "
    "assert torch.cuda.is_available(), 'CUDA unavailable'; "
    "print('PyTorch:', torch.__version__, 'CUDA:', torch.version.cuda, 'GPU:', torch.cuda.get_device_name(0))"],
    cwd=repo, check=True)
for tool in ("ffmpeg", "ffprobe"):
    assert shutil.which(tool), f"Missing {tool}"
stamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")
workspace = Path("/content") / ("reframe-audio-tail-" + stamp)
workspace.mkdir()
backup_dir = Path("/content/drive/MyDrive/reframe-benchmarks") / workspace.name
backup_dir.mkdir(parents=True)
video = workspace / VIDEO_SOURCE.name
shutil.copy2(VIDEO_SOURCE, video)
print("來源複本：", video)
print("Drive 備份：", backup_dir)


## 套用音訊修補

這是 Colab 本機測試修補，會讓 `git status` 顯示 `reframe/video_io.py` 修改。
先核對原始程式指紋；若相同修補已套用，直接沿用。其他版本不會被覆寫。
修補只在兩條 FFmpeg 輸出路徑加入 `-af apad`，繼續以 `-shortest` 限制音訊尾端。


In [ ]:
EXPECTED_MODULES = {
  "reframe/__init__.py": "dd64f6469102c2e91b5c2a1de65b99b58cd72725d41b800303bf36b4a77d5c6b",
  "reframe/__main__.py": "3c621b28c99f727aecd57ebdf86211480f131f32f7620eef8543312d0923d33c",
  "reframe/camera.py": "92ddd58da13ce739e780df5dc5317e4dd659fd8947c2d835aecfa360d2c47fb6",
  "reframe/cli.py": "fbcbbb4a0513e0192b24211b2c1fe9cc339a88e1f47cd3c81a0b06db54ef379b",
  "reframe/config.py": "68f43c6c21a697542a4a4499e8b9203333bd1d7e1db424cc1874551f2284b057",
  "reframe/contracts.py": "1aaa7ac5ed0ba0d3eef789af2c795f38a5e96d4df880f3407dfb90af7ec5cba4",
  "reframe/debug.py": "bc1f63a7020747d8a82011692df24d121fc679d29e194efdfbcdd2d966df289a",
  "reframe/geometry.py": "338e5df9c6087dfc8e693b57731e24d1ca213a97dac39a90a81871df5118ac10",
  "reframe/perception/__init__.py": "5384bfdb2df380b6557cc7a71d16891415bccaa87699406e236f752c6415389f",
  "reframe/perception/motion.py": "69fd50f826defd5e60c93b3b3977560b3cebd7f6c58dccb6645b7f57478ba39b",
  "reframe/perception/pose.py": "a153223ba9baf5d09c9c188f25629fa9b62edb0fa38e5871bb651e5879964545",
  "reframe/perception/segmentation.py": "02abbc8a4045f98f2198b2db7e779c63cb35e1e932d034144cb7b67d68b4c260",
  "reframe/pipeline.py": "1bc6285b852f46f2c758c86944fc943240985e7ba606417a39572578ce7f1441",
  "reframe/precrop.py": "8bc86d818ed91b9ea80ccd01f38cb2155db71f0c07b87fb39e4439b148bc2428",
  "reframe/runtime.py": "826cc12da5426cd7f281df52c10be7675512d8c627bea37c4f51961710d77be6",
  "reframe/saliency/__init__.py": "5384bfdb2df380b6557cc7a71d16891415bccaa87699406e236f752c6415389f",
  "reframe/saliency/backends/__init__.py": "5384bfdb2df380b6557cc7a71d16891415bccaa87699406e236f752c6415389f",
  "reframe/saliency/backends/deepgazemr.py": "75b87d0de7e09dbdc0d80ade65a7c8e55cbe63a26411eb283cbdbc7397750c47",
  "reframe/saliency/backends/handcrafted.py": "46ecf071a2777a6bb4a6c51eaac756c268ea3cc37b057693d640e712a44fcaff",
  "reframe/saliency/base.py": "5f4c465e2253b4f3c704fc3daa9bcbfde50110155989ee621c61b55a94131e17",
  "reframe/saliency/cache.py": "ab34a4de85d23acc24e4567923bc489e3a448070c3a7ed793eb696a03c2233b7",
  "reframe/saliency/cascade.py": "d8b674093a102faf72265b7f91175d2f5a320aaeecc0a85e226af34b1f0c64d3",
  "reframe/saliency/factory.py": "f4bc4a42230bf05ea824e822599ec517fb2918716946765b4ee80bf4cf8b2c3b",
  "reframe/saliency/regions.py": "0f85600ca284e60c464378d9b10b33f9d74d8445d4c4dede054a0d0e27704ddc",
  "reframe/saliency/scheduler.py": "e6502e5000785d0eeeb4d38c324130847612039bb20f84a23235402ef09f9501",
  "reframe/saliency/service.py": "f00412acb826c0762af11de9c8d4fc7f22242e2840a28719fa932ffb5490f6cc",
  "reframe/scenes.py": "528eb31b2d506278c0ba38d7397a38456f7571df5076b2eb7731af549e0b6e4d",
  "reframe/segments.py": "c7676439e801c9e106cb53d1744bdfedd218447355d318ff258f2cfa0b996761",
  "reframe/subjects.py": "c09be0d450a338ae5fab3e2e2d2ed06bcc20751eb58e2f5019bfa77d0fe488ed",
  "reframe/timing.py": "af8206d70d1ba8685858a3d529fcca720a35352d9b1d4ad047e79b1272166eec",
  "scripts/benchmark_colab.py": "60cad122ae950a1efafbf30943d2a09f3c03ede10c9a36448c637b0a1a75f8d6"
}
AUDIO_BEFORE = '83f33d96a23195fcbda81dab51495b7be11805947286355e05e912a250e7ac36'
AUDIO_AFTER = '0257df8ff06bf84286381c235a4a9822cc57278f45ef91b1b813b1c3294e9f67'
AUDIO_PATCH = 'diff --git a/reframe/video_io.py b/reframe/video_io.py\nindex f67558a..418cbd7 100644\n--- a/reframe/video_io.py\n+++ b/reframe/video_io.py\n@@ -194,6 +194,9 @@ def run_ffmpeg_mux(\n                 "aac",\n                 "-b:a",\n                 audio_bitrate,\n+                # Pad short source audio so -shortest ends at rendered video EOF.\n+                "-af",\n+                "apad",\n                 "-shortest",\n             ]\n             if final_path.suffix.lower() in {".mp4", ".mov", ".m4v"}:\n@@ -439,7 +442,9 @@ class DirectVideoWriter(LosslessWriter):\n         ]\n         if vf:\n             cmd += ["-vf", vf]\n-        cmd += ["-c:a", "aac", "-b:a", args.audio_bitrate, "-shortest"]\n+        # Keep every rendered frame even when source audio ends slightly earlier.\n+        # -shortest still bounds audio for --max-frames and other partial renders.\n+        cmd += ["-c:a", "aac", "-b:a", args.audio_bitrate, "-af", "apad", "-shortest"]\n         if self.destination.suffix.lower() in {".mp4", ".mov", ".m4v"}:\n             cmd += ["-movflags", "+faststart"]\n         cmd += [str(self.partial)]\n'

for relative, expected in EXPECTED_MODULES.items():
    path = repo / relative
    if not path.is_file() or hashlib.sha256(path.read_bytes()).hexdigest() != expected:
        raise RuntimeError(f"來源版本不同：{relative}；請保留檔案並先核對版本")
target = repo / "reframe/video_io.py"
current = hashlib.sha256(target.read_bytes()).hexdigest()
if current == AUDIO_BEFORE:
    patchfile = workspace / "audio-tail.patch"
    patchfile.write_text(AUDIO_PATCH)
    subprocess.run(["git", "apply", "--check", str(patchfile)], cwd=repo, check=True)
    subprocess.run(["git", "apply", str(patchfile)], cwd=repo, check=True)
elif current != AUDIO_AFTER:
    raise RuntimeError("video_io.py 不是已知原版或本次修正版；未覆寫")
assert hashlib.sha256(target.read_bytes()).hexdigest() == AUDIO_AFTER
print("音訊修補已確認；推論模組與原基準一致")


## 建立測試 runner

此格內含完整 runner，寫入 `/content`。它重用原有 collector 的幀數、FPS、CUDA、NVENC
及分段驗證，不變更 collector 或 AI 策略。短測用真正的 direct／lossless writer，沒有載入模型。


In [ ]:
%%writefile /content/reframe_audio_tail_test.py
"""One audio-tail regression, preserving the existing gap1 visual baseline.

The companion notebook also embeds this runner for use before a GitHub release.
"""
from __future__ import annotations

import argparse
import hashlib
import importlib.util
import json
import logging
from pathlib import Path
import shutil
import subprocess
import sys
import traceback


def collector(repo):
    sys.path.insert(0, str(repo))
    spec = importlib.util.spec_from_file_location("audio_tail_benchmark", repo / "scripts/benchmark_colab.py")
    module = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(module)
    # Only this process gains a one-case reporting group; the collector is unchanged.
    module.SUITES["audio-tail"] = (module.CASES[0],)
    return module


def endpoints(media):
    result = {}
    for kind in ("video", "audio"):
        stream = next((s for s in media["streams"] if s["codec_type"] == kind), None)
        if stream and stream.get("duration") not in (None, "N/A"):
            result[kind + "_end_seconds"] = float(stream.get("start_time", 0)) + float(stream["duration"])
    if len(result) == 2:
        result["audio_minus_video_end_seconds"] = result["audio_end_seconds"] - result["video_end_seconds"]
    return result


def source_fingerprint(repo):
    paths = [*sorted((repo / "reframe").rglob("*.py")), repo / "scripts/benchmark_colab.py"]
    return {str(path.relative_to(repo)): hashlib.sha256(path.read_bytes()).hexdigest() for path in paths}


def mux_smoke(args, benchmark, results):
    """Exercise both production writers with 120 frames and deliberately short audio."""
    import cv2
    import numpy as np
    from reframe.config import AppConfig
    from reframe.video_io import DirectVideoWriter, LosslessWriter, run_ffmpeg_mux

    source = args.output_dir / "short-audio-source.mp4"
    subprocess.run([
        "ffmpeg", "-hide_banner", "-loglevel", "info", "-n",
        "-f", "lavfi", "-i", "color=c=gray:s=320x240:r=60:d=2",
        "-f", "lavfi", "-i", "sine=frequency=440:sample_rate=44100:duration=1.866666667",
        "-map", "0:v:0", "-map", "1:a:0", "-c:v", "libx264", "-pix_fmt", "yuv420p",
        "-c:a", "libmp3lame", str(source),
    ], check=True)
    original = benchmark.probe(source)
    results["source_probe"] = original
    if endpoints(original).get("audio_minus_video_end_seconds", 0) >= -.05:
        raise ValueError("Synthetic source must have audio shorter than its video")
    config = AppConfig(video_encoder=args.video_encoder, ffmpeg_log_level="info")
    for mode in ("direct", "lossless"):
        output = args.output_dir / f"mux-{mode}.mp4"
        intermediate = args.output_dir / "silent.mkv"
        record = {"mode": mode, "status": "failed", "output": str(output)}
        results["runs"].append(record)
        if mode == "direct":
            writer = DirectVideoWriter(output, source, 60, (320, 240), config)
        else:
            writer = LosslessWriter(str(intermediate), 60, (320, 240), "info")
        try:
            for index in range(120):
                writer.write(np.full((240, 320, 3), 240 if index >= 112 else 40, dtype=np.uint8))
            writer.release()
        finally:
            writer.abort()
        actual = writer.encoder if mode == "direct" else run_ffmpeg_mux(
            str(intermediate), str(source), str(output), args.video_encoder,
            config.audio_bitrate, config.crf, config.preset_ffmpeg, None,
            ffmpeg_log_level="info")
        record["actual_encoder"] = actual
        if actual != args.video_encoder:
            raise ValueError(f"Encoder fallback: requested {args.video_encoder}, got {actual}")
        record["output_probe"] = benchmark.probe(output)
        record["validation"] = benchmark.validate_output(original, record["output_probe"],
            {"frames_processed": 120, "output_width": 320, "output_height": 240}, None)
        record["endpoints"] = endpoints(record["output_probe"])
        cap = cv2.VideoCapture(str(output))
        means = []
        try:
            while True:
                ok, frame = cap.read()
                if not ok:
                    break
                means.append(float(frame.mean()))
        finally:
            cap.release()
        if len(means) != 120 or not all(value > 220 for value in means[-8:]) or means[-9] >= 60:
            raise ValueError("The distinctive last eight video frames were not retained")
        record["tail_marker_verified"] = True
        record["status"] = "ok"
        print(json.dumps(record, ensure_ascii=False, indent=2), flush=True)
    intermediate.unlink(missing_ok=True)


def main(argv=None):
    parser = argparse.ArgumentParser(description=__doc__)
    parser.add_argument("--repo", type=Path, default=Path(__file__).resolve().parents[1])
    parser.add_argument("--mode", choices=("mux", "full"), required=True)
    parser.add_argument("--input", type=Path)
    parser.add_argument("--output-dir", type=Path, required=True)
    parser.add_argument("--backup-dir", type=Path)
    parser.add_argument("--seg-model", default="yolo26n-seg.pt")
    parser.add_argument("--pose-model", default="yolo26n-pose.pt")
    parser.add_argument("--device", default="0")
    parser.add_argument("--saliency-device", default="cuda")
    parser.add_argument("--video-encoder", default="hevc_nvenc")
    args = parser.parse_args(argv)
    if args.mode == "full" and (args.input is None or not args.input.is_file()):
        parser.error("--mode full requires an existing --input video")
    args.repo, args.output_dir = args.repo.resolve(), args.output_dir.resolve()
    if args.input:
        args.input = args.input.resolve()
    args.output_dir.mkdir(parents=True, exist_ok=False)
    args.max_frames, args.stats_interval, args.log_level = None, 5.0, "INFO"
    logging.basicConfig(level=logging.INFO)
    benchmark = collector(args.repo)
    results = {"schema_version": 3, "test": "audio-tail", "status": "running",
               "environment": benchmark.environment(), "source_sha256": source_fingerprint(args.repo),
               "configuration": {**{key: str(value) if isinstance(value, Path) else value
                                     for key, value in vars(args).items()}, "suite": "audio-tail"},
               "runs": []}
    results["runner_sha256"] = hashlib.sha256(Path(__file__).read_bytes()).hexdigest()
    results["audio_patch_diff"] = subprocess.run(
        ["git", "diff", "HEAD", "--", "reframe/video_io.py"], cwd=args.repo,
        check=True, capture_output=True, text=True).stdout
    code = 1
    try:
        if args.mode == "mux":
            mux_smoke(args, benchmark, results)
        else:
            results["source_probe"] = benchmark.probe(args.input)
            case = benchmark.CASES[0]
            results["workload"] = benchmark.workload(results["source_probe"], args, [(case, False, 1)])
            print("Exactly one full gap1 baseline, no model warmups:", results["workload"], flush=True)
            benchmark.save(args.output_dir, results)
            record = benchmark.run_one(args, case, False, 1, results["source_probe"])
            results["runs"].append(record)
            if "output_probe" in record:
                record["endpoints"] = endpoints(record["output_probe"])
            if record["status"] != "ok":
                raise RuntimeError(record["error"])
        results["status"] = "ok"
        code = 0
    except (Exception, KeyboardInterrupt) as error:
        results["status"] = "failed"
        results["error"] = f"{type(error).__name__}: {error}"
        traceback.print_exc()
    finally:
        if args.mode == "full":
            benchmark.save(args.output_dir, results)
        else:
            (args.output_dir / "results.json").write_text(json.dumps(results, ensure_ascii=False, indent=2))
        archive = benchmark.archive_results(args.output_dir)
        print("Diagnostic ZIP:", archive, flush=True)
        if args.backup_dir:
            args.backup_dir.mkdir(parents=True, exist_ok=True)
            shutil.copy2(archive, args.backup_dir / archive.name)
            # Keep the produced video even if a later metadata check failed.
            if args.mode == "full":
                for path in args.output_dir.glob("*.mp4"):
                    shutil.copy2(path, args.backup_dir / path.name)
            print("Drive backup:", args.backup_dir, flush=True)
    return code


if __name__ == "__main__":
    raise SystemExit(main())


## 執行短測，再跑一次完整影片

短測必須保留 120 幀、60 FPS、最後八幀亮色標記，且實際編碼器為 `hevc_nvenc`。
通過才開始完整影片。輸出與原始 log 寫在 `/content`，結束或可捕捉失敗時備份 ZIP、log、MP4 到 Drive。
中途直接刪除 runtime 無法保證執行備份；執行時保持 Colab session。


In [ ]:
runner = Path("/content/reframe_audio_tail_test.py")
mux_dir, run_dir = workspace / "mux-smoke", workspace / "full-baseline"
common = [sys.executable, "-u", str(runner), "--repo", str(repo),
          "--video-encoder", "hevc_nvenc", "--backup-dir", str(backup_dir)]

def run_logged(mode, directory, extra=()):
    command = common + ["--mode", mode, "--output-dir", str(directory), *extra]
    log = workspace / (directory.name + "-runner.log")
    process = None
    try:
        with log.open("xb") as handle:
            process = subprocess.Popen(command, cwd=repo, stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
            try:
                while chunk := process.stdout.read1(65536):
                    handle.write(chunk)
                    handle.flush()
                    print(chunk.decode("utf-8", errors="replace"), end="", flush=True)
                code = process.wait()
            except BaseException:
                process.terminate()
                try:
                    process.wait(timeout=10)
                except subprocess.TimeoutExpired:
                    process.kill()
                    process.wait()
                raise
        if code:
            raise subprocess.CalledProcessError(code, command)
    finally:
        # Include the runner's own native FFmpeg errors in the diagnostic archive.
        if directory.is_dir() and log.is_file():
            shutil.copy2(log, directory / "runner.log")
            archive = Path(str(directory) + ".zip")
            # Runner already archived before exiting; add its enclosing log exactly once.
            with zipfile.ZipFile(archive, "a", compression=zipfile.ZIP_DEFLATED) as bundle:
                if "runner.log" not in bundle.namelist():
                    bundle.write(log, arcname="runner.log")
                for path in directory.iterdir():
                    if path.is_file() and path.suffix in {".json", ".csv", ".jsonl", ".log"}:
                        if path.name not in bundle.namelist():
                            bundle.write(path, arcname=path.name)
            shutil.copy2(archive, backup_dir / archive.name)
            for path in directory.glob("*.mp4"):
                if mode == "full":
                    shutil.copy2(path, backup_dir / path.name)
        if log.is_file():
            shutil.copy2(log, backup_dir / log.name)
        print("\n本機結果：", directory, "\nDrive 備份：", backup_dir)

run_logged("mux", mux_dir)
smoke = json.loads((mux_dir / "results.json").read_text())
assert smoke["status"] == "ok"
print("短測通過，開始一次完整 gap1 基準")
run_logged("full", run_dir, ["--input", str(video),
    "--seg-model", str(SEG_MODEL), "--pose-model", str(POSE_MODEL),
    "--device", "0", "--saliency-device", "cuda"])


## 結果與變暗／人物交錯片段對照

同一支原影片應為 **10,252 幀、60 FPS**；實際目標由來源 ffprobe 決定。
驗收保留完整畫面與音訊，不要求 DeepGaze 呼叫減少。FFmpeg／AAC 可能留下短的緩衝音訊尾端，
下表列出音畫結束時間差；仍請聽看最後一秒。

`locked_subject_missing` 是模型決策紀錄，不代表視覺焦點遺失，也不能單憑該欄證明變暗是原因。
每五秒統計無法定位逐幀 ID；可在 `EVENT_WINDOWS` 填入您看到變暗或交錯的起訖秒數，篩選重疊分段。


In [ ]:
import pandas as pd
from IPython.display import display

result = json.loads((run_dir / "results.json").read_text())
print("測試狀態：", result["status"])
if result.get("error"):
    print(result["error"])
for record in result["runs"]:
    print("輸出：", record["output"])
    print("影片驗證：", json.dumps(record.get("validation", {}), ensure_ascii=False, indent=2))
    print("模型裝置：", json.dumps(record.get("device_validation", {}), ensure_ascii=False, indent=2))
    print("音畫結束时间：", json.dumps(record.get("endpoints", {}), ensure_ascii=False, indent=2))
    summary = record.get("summary", {})
    print("處理秒數：", summary.get("elapsed_seconds"),
          "DeepGaze 次數：", summary.get("saliency_actual_forward_calls"))
    rows = [row for row in record.get("segments", [])]
    if rows:
        table = pd.json_normalize(rows)
        columns = ["start_seconds", "end_seconds", "scene_index", "frames",
                   "seg_detector_calls", "subject_switches", "frames_with_subject",
                   "saliency_tier_reasons.locked_subject_missing",
                   "saliency_actual_forward_calls", "processing_wall_seconds"]
        missing = "saliency_tier_reasons.locked_subject_missing"
        if missing not in table:
            table[missing] = 0
        table[missing] = table[missing].fillna(0).astype(int)
        columns = [name for name in columns if name in table]
        display(table[columns])
        EVENT_WINDOWS = []  # 例如 [(32.0, 35.0, "短暫變暗"), (80.0, 86.0, "人物交錯")]
        for start, end, note in EVENT_WINDOWS:
            assert end > start >= 0
            print(f"{note}：{start}～{end} 秒（顯示重疊的統計段）")
            display(table.loc[(table.start_seconds < end) & (table.end_seconds > start), columns])
print("請保留 Drive 中的 mux-smoke.zip、full-baseline.zip 與 gap1-baseline-run1.mp4：", backup_dir)
